# Joins in Spark - Inner, Outer, Left, Right

Connecting datasets and understanding how Spark handles relationships
In real-world data engineering, you’ll rarely find all the information you need in a single dataset. A sales table might tell you how much revenue was generated, but you’ll need a products table to know what was sold, and a customers table to know who bought it.

This is where joins come in. A join is simply the act of connecting two datasets using a key they share. In Spark, joins are incredibly powerful — but they can also be confusing at first, because different join types behave differently.

Today, we’ll explore four of the most common joins:

- Inner Join
- Left Join
- Right Join
- Full Outer Join

And we’ll do it step by step with examples you can run yourself.

Our Example Datasets
Let’s create two small DataFrames so we can see what’s happening clearly.

## Customers Table

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("day10-demo").getOrCreate()

customers = [
    (1, "Alice", "North"),
    (2, "Bob", "South"),
    (3, "Charlie", "East"),
    (4, "David", "West")
]

df_customers = spark.createDataFrame(customers, ["cust_id", "name", "region"])
df_customers.show()

# Orders Table

In [0]:
orders = [
    (101, 1, 250),   # order_id, cust_id, amount
    (102, 2, 450),
    (103, 5, 300),   # customer_id 5 doesn't exist in customers
]

df_orders = spark.createDataFrame(orders, ["order_id", "cust_id", "amount"])
df_orders.show()

Notice something important here:

- Customers 1 and 2 placed orders.
- Customer 5 placed an order, but we don’t have their details in the customers table.
- Customers 3 and 4 exist in the customers table but didn’t place any orders.

This setup will make it clear how different joins behave.

# Inner Join

An inner join returns only the rows where a match exists in both tables.

In [0]:
df_inner = df_customers.join(df_orders, on="cust_id", how="inner")
df_inner.show()

Explanation:

- Customer 1 and 2 exist in both datasets → included.
- Customer 3 and 4 didn’t order → excluded.
- Order for customer 5 has no customer record → excluded.

Use inner joins when you only want the intersection of data.

## Left Join

A left join keeps all rows from the left DataFrame and adds matches from the right. If no match is found, nulls are filled in.

In [0]:
df_left = df_customers.join(df_orders, on="cust_id", how="left")
df_left.show()

Explanation:

- All customers are kept.
- Customers 3 and 4 have no orders → null values for order details.
- Order for customer 5 is ignored because customer 5 doesn’t exist in the left table.

Use left joins when your left dataset is primary, and you want to preserve it regardless of matches.

## Right Join

In [0]:
df_right = df_customers.join(df_orders, on="cust_id", how="right")
df_right.show()

Explanation:

- All orders are kept.
- Order 103 with cust_id 5 has no customer match → nulls for customer details.
- Customers 3 and 4 disappear, since they had no orders.

Use right joins when your right dataset (orders in this case) is primary.

## Full Outer Join

In [0]:
df_outer = df_customers.join(df_orders, on="cust_id", how="outer")
df_outer.show()

Explanation:
    
- Everyone is included, regardless of match.
- Unmatched customers → null orders.
- Unmatched order → null customer details.

Use full outer joins when you want the union of everything and are okay handling nulls.

### Why Joins Matter in Data Engineering
Let’s tie this back to real-world data pipelines. Imagine:

- Customers table → master data from CRM.
- Orders table → transaction data from e-commerce.
- If you run an inner join, you’ll only analyze customers who placed orders.
- A left join helps you answer, “Which customers exist but haven’t ordered yet?”
- A right join helps you track, “Which orders came from unregistered or missing customers?”
- A full outer join gives you a complete diagnostic view, including mismatches.

### Practical Notes for Freshers

**Column names**: If join columns have different names, use on=[df_customers.cust_id == df_orders.customer_id].

**Data skew**: Large joins can be expensive. Spark may shuffle terabytes of data across the cluster. Later, we’ll learn about techniques like broadcast joins and bucketing to optimize performance.

**Null handling**: Joins often produce nulls; be ready to clean them with na.fill() or conditional logic.

### Wrapping Up
Today we dove into joins, one of the most important tools in your Spark toolkit. You learned how:

- Inner joins keep only matching rows.
- Left joins preserve all rows from the left.
- Right joins preserve all rows from the right.
- Full outer joins include everything, filling gaps with nulls.

Each join type answers a different business question. The real skill as a data engineer is not just writing the join, but choosing the right one for the problem.

That’s Day 10. You now understand Spark joins in detail.

Next up, Day 11: Handling Missing Data in Spark — Dealing with Nulls using na.drop, na.fill, and na.replace. We’ll learn how to clean data efficiently before passing it downstream.